In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("data")
if not data_dir.exists():
    data_dir = Path("../data")

chem = pd.read_csv(data_dir / "1.csv")
sensory = pd.read_csv(data_dir / "4.csv")

feature_cols = chem.columns.drop(
    ["beer", "beer_id", "tasting_category_fine"]
)

print("候选输入数量：", len(feature_cols))
for i, name in enumerate(feature_cols, start=1):
    print(f"{i:3d}. {name}")

候选输入数量： 231
  1. acetaldehyde
  2. CS2
  3. DES
  4. DMS
  5. ethyl_2.methyl_butyrate
  6. ethyl_acetate
  7. ethyl_decanoate
  8. ethyl_hexanoate
  9. ethyl_isovalerate
 10. ethyl_octanoate
 11. ethyl_propionate
 12. H2S
 13. isoamyl_alcohol
 14. isobutanol
 15. isobutyl_acetate
 16. isopentyl_acetate
 17. MeSH
 18. octyl_acetate
 19. phenethyl_acetate
 20. phenethyl_alcohol
 21. X4VG
 22. SO2.mg.L.
 23. acetic_acid.g.L.
 24. ammonia.mg.L.
 25. color.EBC.
 26. betaglucan.mg.L.
 27. lactic_acid.mg.L.
 28. glycerol
 29. iron
 30. pH
 31. protein.g.L.
 32. bitterness
 33. GFS
 34. ethanol..v.v.
 35. CO2.PSI.
 36. unfermentables
 37. kcalperc
 38. X1.propanol
 39. X2.3.butanedione
 40. X3.methylbutanal
 41. propylene.glycol
 42. propyl.acetate
 43. n.pentanol
 44. X2.methyl.1.butanol
 45. isopentyl.formate
 46. ethyl.isobutyrate
 47. X3.methyl.2.butenal
 48. X2.methyltetrahydrofuran.3.one
 49. ethyl.butyrate
 50. butyl.acetate
 51. methylpyrazine
 52. furfural
 53. ethyl.lactate
 54. X3Z.

In [2]:
summary_cols = [
    "esters_sum", "aroma_hops_sum", "acids_sum", "sulfur_sum"
]

property_cols = [
    "pH", "color.EBC.", "bitterness", "CO2.PSI.", "kcalperc"
]

pending_cols = [
    "GFS", "unfermentables", "protein.g.L.", "betaglucan.mg.L."
]

def classify_feature(name):
    if name in summary_cols:
        return "汇总指标"
    if name in property_cols:
        return "理化或整体测量"
    if name in pending_cols:
        return "混合成分或定义待核实"
    return "单项化学测量候选"

feature_roles = pd.DataFrame({
    "字段": feature_cols,
    "暂定角色": [classify_feature(c) for c in feature_cols]
})

compound_cols = feature_roles.loc[
    feature_roles["暂定角色"] == "单项化学测量候选", "字段"
].tolist()

print(feature_roles["暂定角色"].value_counts())
print("\n单项化学测量候选数量：", len(compound_cols))

暂定角色
单项化学测量候选      218
理化或整体测量         5
混合成分或定义待核实      4
汇总指标            4
Name: count, dtype: int64

单项化学测量候选数量： 218


In [3]:
import numpy as np

compound_data = chem.set_index("beer")[compound_cols]

# 至少有200款共同有效样本，才计算该指标对的相关性
corr = compound_data.corr(
    method="spearman",
    min_periods=200
)

# 只保留上三角：排除自己与自己的相关性，以及重复组合
upper_mask = np.triu(
    np.ones(corr.shape, dtype=bool),
    k=1
)

pairs = (
    corr.where(upper_mask)
    .stack()
    .rename("Spearman相关系数")
    .reset_index()
)

pairs.columns = ["指标1", "指标2", "Spearman相关系数"]
pairs["相关强度"] = pairs["Spearman相关系数"].abs()
pairs = pairs.sort_values("相关强度", ascending=False)

print("成功计算的指标对数量：", len(pairs))
print("\n相关强度最高的10对：")
print(pairs.head(10).round(3).to_string(index=False))

成功计算的指标对数量： 47524

相关强度最高的10对：
                     指标1                                  指标2  Spearman相关系数  相关强度
        pentyl.octanoate             X2.methylbutyl.octanoate         0.937 0.937
     X2.methyl.1.butanol                    isopentyl.formate         0.922 0.922
     X.Z..beta.farnesene                     pentyl.octanoate         0.916 0.916
        pentyl.octanoate X1.2.dihydro.8.hydroxy..2E..linalool         0.912 0.912
X2.phenyl.ethyl.butyrate                     pentyl.octanoate         0.905 0.905
    ethyl.tetradecanoate                  ethyl.hexadecanoate         0.903 0.903
     X.Z..beta.farnesene             X2.methylbutyl.octanoate         0.899 0.899
X2.phenyl.ethyl.butyrate                  X.Z..beta.farnesene         0.893 0.893
             E.nerolidol      dihydroisocaryophyllene.epoxide         0.886 0.886
     isopentyl.hexanoate                     pentyl.octanoate         0.882 0.882


In [4]:
pairs = (
    corr.where(upper_mask)
    .stack()
    .dropna()  # 明确删除被遮掉的位置和无法计算的相关值
    .rename("Spearman相关系数")
    .reset_index()
)

pairs.columns = ["指标1", "指标2", "Spearman相关系数"]
pairs["相关强度"] = pairs["Spearman相关系数"].abs()
pairs = pairs.sort_values("相关强度", ascending=False)

expected_max = len(compound_cols) * (len(compound_cols) - 1) // 2

print("最多应有的指标对：", expected_max)
print("实际有效指标对：", len(pairs))
print("相关系数缺失数：", pairs["Spearman相关系数"].isna().sum())
print("自己与自己配对数：", (pairs["指标1"] == pairs["指标2"]).sum())

assert len(pairs) <= expected_max
assert not (pairs["指标1"] == pairs["指标2"]).any()

最多应有的指标对： 23653
实际有效指标对： 23653
相关系数缺失数： 0
自己与自己配对数： 0


In [5]:
rows = []
n_nodes = len(compound_cols)
max_edges = n_nodes * (n_nodes - 1) // 2

for threshold in [0.6, 0.7, 0.8, 0.9]:
    edges = pairs.loc[pairs["相关强度"] >= threshold]

    # 至少连接了一条边的指标
    connected = set(edges["指标1"]) | set(edges["指标2"])

    rows.append({
        "阈值": threshold,
        "边数": len(edges),
        "正相关边": (edges["Spearman相关系数"] > 0).sum(),
        "负相关边": (edges["Spearman相关系数"] < 0).sum(),
        "有连接的节点数": len(connected),
        "孤立节点数": n_nodes - len(connected),
        "网络密度": len(edges) / max_edges
    })

network_summary = pd.DataFrame(rows)
print(network_summary.round(4).to_string(index=False))

 阈值  边数  正相关边  负相关边  有连接的节点数  孤立节点数   网络密度
0.6 245   242     3      120     98 0.0104
0.7  97    97     0       69    149 0.0041
0.8  34    34     0       30    188 0.0014
0.9   6     6     0        9    209 0.0003


In [7]:
import networkx as nx

threshold = 0.6
network_edges = pairs.loc[
    pairs["相关强度"] >= threshold
].copy()

G = nx.Graph()

# 先加入全部节点，保留孤立节点
G.add_nodes_from(compound_cols)

for _, row in network_edges.iterrows():
    G.add_edge(
        row["指标1"],
        row["指标2"],
        rho=row["Spearman相关系数"],
        weight=row["相关强度"]
    )

node_stats = pd.DataFrame({
    "指标": list(G.nodes()),
    "连接数": [G.degree(node) for node in G.nodes()],
    "关联强度之和": [
        G.degree(node, weight="weight")
        for node in G.nodes()
    ]
}).sort_values(
    ["连接数", "关联强度之和"],
    ascending=False
)

components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True
)

print("节点数：", G.number_of_nodes())
print("边数：", G.number_of_edges())
print("连通分量数（含孤立节点）：", len(components))
print("最大的5个连通分量大小：",
      [len(group) for group in components[:5]])

print("\n连接数最多的10个节点：")
print(node_stats.head(10).round(3).to_string(index=False))

节点数： 218
边数： 245
连通分量数（含孤立节点）： 108
最大的5个连通分量大小： [96, 5, 5, 2, 2]

连接数最多的10个节点：
                                     指标  连接数  关联强度之和
               X2.methylbutyl.octanoate   16  12.029
                       pentyl.octanoate   14  11.109
   X1.2.dihydro.8.hydroxy..2E..linalool   14  10.572
                    isopentyl.hexanoate   14  10.462
                      X2.butyloctanoate   13   9.416
   trans.dihydro.alpha.terpinyl.acetate   13   9.289
caryophylla.4.12..8.13..dien.5.alpha.ol   13   8.459
                    X.Z..beta.farnesene   12   9.450
                   ethyl.tetradecanoate   12   9.099
                   ethyl.pentadecanoate   12   8.860


In [8]:
rng = np.random.default_rng(42)

n_boot = 100
n_samples = len(compound_data)

# 当前245条边在相关矩阵中的位置
col_position = {
    name: i for i, name in enumerate(compound_data.columns)
}

left = network_edges["指标1"].map(col_position).to_numpy()
right = network_edges["指标2"].map(col_position).to_numpy()
original_sign = np.sign(
    network_edges["Spearman相关系数"].to_numpy()
)

stable_count = np.zeros(len(network_edges), dtype=int)

for b in range(n_boot):
    positions = rng.integers(0, n_samples, size=n_samples)
    sampled = compound_data.iloc[positions]

    boot_corr = sampled.corr(
        method="spearman",
        min_periods=200
    ).to_numpy()

    values = boot_corr[left, right]

    # 同时要求：相关强度仍≥0.6，且正负方向不变
    retained = (
        np.isfinite(values)
        & (np.abs(values) >= threshold)
        & (np.sign(values) == original_sign)
    )
    stable_count += retained

    if (b + 1) % 20 == 0:
        print(f"已完成 {b + 1}/{n_boot} 轮")

edge_stability = network_edges.copy()
edge_stability["重抽样保留率"] = stable_count / n_boot

print("\n边的保留率分布：")
print(
    edge_stability["重抽样保留率"]
    .describe()
    .round(3)
    .to_string()
)

print("\n保留率≥80%的边数：",
      (edge_stability["重抽样保留率"] >= 0.8).sum())

已完成 20/100 轮
已完成 40/100 轮
已完成 60/100 轮
已完成 80/100 轮
已完成 100/100 轮

边的保留率分布：
count    245.000
mean       0.867
std        0.166
min        0.470
25%        0.740
50%        0.960
75%        1.000
max        1.000

保留率≥80%的边数： 172


In [9]:
# 保留所有节点，只筛选边
stable_edges = edge_stability.loc[
    edge_stability["重抽样保留率"] >= 0.8
].copy()

G_stable = nx.Graph()
G_stable.add_nodes_from(compound_cols)

for _, row in stable_edges.iterrows():
    G_stable.add_edge(
        row["指标1"],
        row["指标2"],
        rho=float(row["Spearman相关系数"]),
        weight=float(row["相关强度"]),
        retention=float(row["重抽样保留率"])
    )

output_dir = data_dir.resolve().parent / "outputs"
output_dir.mkdir(exist_ok=True)

feature_roles.to_csv(
    output_dir / "feature_roles.csv",
    index=False, encoding="utf-8-sig"
)
edge_stability.to_csv(
    output_dir / "network_edges_bootstrap.csv",
    index=False, encoding="utf-8-sig"
)
stable_edges.to_csv(
    output_dir / "network_edges_stable.csv",
    index=False, encoding="utf-8-sig"
)

nx.write_graphml(
    G_stable,
    output_dir / "compound_network_stable.graphml"
)

print("稳定子网络节点数：", G_stable.number_of_nodes())
print("稳定子网络边数：", G_stable.number_of_edges())
print("稳定子网络孤立节点数：",
      nx.number_of_isolates(G_stable))
print("文件保存位置：", output_dir)

稳定子网络节点数： 218
稳定子网络边数： 172
稳定子网络孤立节点数： 122
文件保存位置： C:\Users\Lenovo\PycharmProjects\PythonProject2\outputs


In [10]:
# 按beer编号对齐化学数据和感官评分
sensory_aligned = sensory.set_index("beer").reindex(
    compound_data.index
)

assert sensory["beer"].is_unique
assert set(compound_data.index) == set(sensory["beer"])

target_cols = [
    col for col in sensory.columns
    if col.startswith(("A_", "F_"))
]

association_rows = []

for target in target_cols:
    y_target = sensory_aligned[target]

    for compound in compound_cols:
        x_compound = compound_data[compound]

        # 每对变量仅使用双方都有数据的啤酒
        valid = x_compound.notna() & y_target.notna()
        n_valid = int(valid.sum())

        rho = (
            x_compound[valid].corr(
                y_target[valid], method="spearman"
            )
            if n_valid >= 200 else float("nan")
        )

        association_rows.append({
            "感官指标": target,
            "化学指标": compound,
            "有效样本数": n_valid,
            "Spearman相关系数": rho
        })

associations = pd.DataFrame(association_rows)
associations["相关强度"] = (
    associations["Spearman相关系数"].abs()
)

ranked = (
    associations.dropna(subset=["Spearman相关系数"])
    .sort_values(
        ["感官指标", "相关强度"],
        ascending=[True, False]
    )
)

top3_corr = ranked.groupby("感官指标", sort=False).head(3)

associations.to_csv(
    output_dir / "compound_sensory_correlations.csv",
    index=False, encoding="utf-8-sig"
)
top3_corr.to_csv(
    output_dir / "top3_correlation_candidates.csv",
    index=False, encoding="utf-8-sig"
)

# 先展示三个整体香气指标，避免输出过长
show_targets = ["A_malt_all", "A_hops_all", "A_esters_all"]

print(
    top3_corr.loc[
        top3_corr["感官指标"].isin(show_targets)
    ].round(3).to_string(index=False)
)
print("\nTop-3结果总行数：", len(top3_corr))

        感官指标            化学指标  有效样本数  Spearman相关系数  相关强度
A_esters_all       n.hexanol    250         0.357 0.357
A_esters_all    acetaldehyde    250         0.353 0.353
A_esters_all   ethyl_acetate    250         0.321 0.321
  A_hops_all ethyl_octanoate    250         0.464 0.464
  A_hops_all  X.E..2.nonenal    250         0.432 0.432
  A_hops_all ethyl_decanoate    250         0.428 0.428
  A_malt_all    acetaldehyde    250         0.361 0.361
  A_malt_all  methylpyrazine    238         0.356 0.356
  A_malt_all   ethanol..v.v.    250         0.335 0.335

Top-3结果总行数： 90


In [11]:
rng = np.random.default_rng(42)
n_boot = 100

# 行已经按beer对齐；这里只取30项感官目标
combined = pd.concat(
    [compound_data, sensory_aligned[target_cols]],
    axis=1
)

# 对全部218个候选统计，避免只检查原先的前三名
selection_counts = pd.DataFrame(
    0, index=compound_cols, columns=target_cols
)

for b in range(n_boot):
    positions = rng.integers(0, len(combined), size=len(combined))
    sampled = combined.iloc[positions]

    # 提取“化学指标 × 感官指标”这一块相关矩阵
    cross_corr = sampled.corr(
        method="spearman",
        min_periods=200
    ).loc[compound_cols, target_cols]

    for target in target_cols:
        selected = (
            cross_corr[target]
            .dropna()
            .abs()
            .nlargest(3)
            .index
        )
        selection_counts.loc[selected, target] += 1

    if (b + 1) % 20 == 0:
        print(f"已完成 {b + 1}/{n_boot} 轮")

selection_frequency = selection_counts / n_boot

# 给原来的90条候选添加入选频率
top3_stability = top3_corr.copy()
top3_stability["Top3入选频率"] = [
    selection_frequency.loc[compound, target]
    for target, compound in zip(
        top3_stability["感官指标"],
        top3_stability["化学指标"]
    )
]

top3_stability.to_csv(
    output_dir / "top3_correlation_stability.csv",
    index=False, encoding="utf-8-sig"
)

# 同时保存全部候选，方便发现其他稳定入选的指标
selection_frequency.to_csv(
    output_dir / "all_candidates_top3_frequency.csv",
    encoding="utf-8-sig"
)

print(
    top3_stability.loc[
        top3_stability["感官指标"].isin(show_targets)
    ].round(3).to_string(index=False)
)

已完成 20/100 轮
已完成 40/100 轮
已完成 60/100 轮
已完成 80/100 轮
已完成 100/100 轮
        感官指标            化学指标  有效样本数  Spearman相关系数  相关强度  Top3入选频率
A_esters_all       n.hexanol    250         0.357 0.357      0.71
A_esters_all    acetaldehyde    250         0.353 0.353      0.70
A_esters_all   ethyl_acetate    250         0.321 0.321      0.33
  A_hops_all ethyl_octanoate    250         0.464 0.464      0.82
  A_hops_all  X.E..2.nonenal    250         0.432 0.432      0.41
  A_hops_all ethyl_decanoate    250         0.428 0.428      0.40
  A_malt_all    acetaldehyde    250         0.361 0.361      0.63
  A_malt_all  methylpyrazine    238         0.356 0.356      0.52
  A_malt_all   ethanol..v.v.    250         0.335 0.335      0.35


In [12]:
stability_rows = []

for target in target_cols:
    # 按重抽样入选频率排序，取前5个
    top5 = selection_frequency[target].nlargest(5)

    for compound, frequency in top5.items():
        stability_rows.append({
            "感官指标": target,
            "化学指标": compound,
            "Top3入选频率": frequency
        })

stable_candidates = pd.DataFrame(stability_rows)

# 补回原始相关系数，方便同时比较强度、方向和稳定性
stable_candidates = stable_candidates.merge(
    associations[
        ["感官指标", "化学指标",
         "Spearman相关系数", "有效样本数"]
    ],
    on=["感官指标", "化学指标"],
    how="left",
    validate="one_to_one"
)

stable_candidates.to_csv(
    output_dir / "top5_stability_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    stable_candidates.loc[
        stable_candidates["感官指标"].isin(show_targets)
    ].round(3).to_string(index=False)
)

        感官指标                化学指标  Top3入选频率  Spearman相关系数  有效样本数
  A_malt_all        acetaldehyde      0.63         0.361    250
  A_malt_all      methylpyrazine      0.52         0.356    238
  A_malt_all       ethanol..v.v.      0.35         0.335    250
  A_malt_all       terpinen.4.ol      0.26        -0.314    250
  A_malt_all   beta.phellandrene      0.24        -0.323    250
  A_hops_all     ethyl_octanoate      0.82         0.464    250
  A_hops_all      X.E..2.nonenal      0.41         0.432    250
  A_hops_all     ethyl_decanoate      0.40         0.428    250
  A_hops_all            linalool      0.36         0.412    250
  A_hops_all humulene.epoxide.II      0.21         0.405    250
A_esters_all           n.hexanol      0.71         0.357    250
A_esters_all        acetaldehyde      0.70         0.353    250
A_esters_all       ethyl_acetate      0.33         0.321    250
A_esters_all     isoamyl_alcohol      0.25         0.298    250
A_esters_all   lactic_acid.mg.L.      0.

In [13]:
styles = chem.set_index("beer")[
    "tasting_category_fine"
].reindex(compound_data.index)

def style_adjusted_corr(compound, target):
    data = pd.DataFrame({
        "x": compound_data[compound],
        "y": sensory_aligned[target],
        "style": styles
    }).dropna()

    # 对这一对变量的共同有效样本分别排名
    ranks = data[["x", "y"]].rank(method="average")

    # 扣除风格组平均排名，移除组间平均差异
    residuals = ranks - ranks.groupby(
        data["style"]
    ).transform("mean")

    return residuals["x"].corr(residuals["y"])

style_check = stable_candidates.copy()

style_check["控制风格后相关"] = [
    style_adjusted_corr(compound, target)
    for compound, target in zip(
        style_check["化学指标"],
        style_check["感官指标"]
    )
]

style_check.to_csv(
    output_dir / "candidate_style_sensitivity.csv",
    index=False, encoding="utf-8-sig"
)

print(
    style_check.loc[
        style_check["感官指标"].isin(show_targets),
        ["感官指标", "化学指标", "Spearman相关系数",
         "控制风格后相关", "Top3入选频率"]
    ].round(3).to_string(index=False)
)

        感官指标                化学指标  Spearman相关系数  控制风格后相关  Top3入选频率
  A_malt_all        acetaldehyde         0.361    0.032      0.63
  A_malt_all      methylpyrazine         0.356    0.166      0.52
  A_malt_all       ethanol..v.v.         0.335   -0.103      0.35
  A_malt_all       terpinen.4.ol        -0.314   -0.170      0.26
  A_malt_all   beta.phellandrene        -0.323   -0.175      0.24
  A_hops_all     ethyl_octanoate         0.464    0.287      0.82
  A_hops_all      X.E..2.nonenal         0.432    0.230      0.41
  A_hops_all     ethyl_decanoate         0.428    0.237      0.40
  A_hops_all            linalool         0.412    0.278      0.36
  A_hops_all humulene.epoxide.II         0.405    0.158      0.21
A_esters_all           n.hexanol         0.357    0.275      0.71
A_esters_all        acetaldehyde         0.353    0.301      0.70
A_esters_all       ethyl_acetate         0.321    0.238      0.33
A_esters_all     isoamyl_alcohol         0.298    0.263      0.25
A_esters_a

In [14]:
final_candidates = top3_stability.merge(
    style_check[
        ["感官指标", "化学指标", "控制风格后相关"]
    ],
    on=["感官指标", "化学指标"],
    how="left",
    validate="one_to_one"
)

# 原始Top-3不一定都进入“频率Top-5”，补算未匹配的项
missing = final_candidates["控制风格后相关"].isna()

for idx in final_candidates.index[missing]:
    row = final_candidates.loc[idx]
    final_candidates.loc[idx, "控制风格后相关"] = (
        style_adjusted_corr(row["化学指标"], row["感官指标"])
    )

final_candidates["方向一致"] = (
    np.sign(final_candidates["Spearman相关系数"])
    == np.sign(final_candidates["控制风格后相关"])
)

final_candidates.to_csv(
    output_dir / "top3_candidate_evidence.csv",
    index=False,
    encoding="utf-8-sig"
)

print("候选记录数：", len(final_candidates))
print("涉及感官指标数：", final_candidates["感官指标"].nunique())
print("调整后相关系数缺失数：",
      final_candidates["控制风格后相关"].isna().sum())
print("文件位置：", output_dir / "top3_candidate_evidence.csv")

候选记录数： 90
涉及感官指标数： 30
调整后相关系数缺失数： 0
文件位置： C:\Users\Lenovo\PycharmProjects\PythonProject2\outputs\top3_candidate_evidence.csv


In [15]:
official_corr = pd.read_csv(
    data_dir / "2.csv",
    index_col=0
)

print("官方相关矩阵大小：", official_corr.shape)
print("行列名称顺序一致：",
      official_corr.index.equals(official_corr.columns))

print("\n前10个官方名称：")
print(official_corr.columns[:10].tolist())

# 查附件8是否提供化学名称对应信息
extra_book = pd.ExcelFile(data_dir / "8.xlsx")
threshold_table = pd.read_excel(
    extra_book,
    sheet_name="Sensory thresholds"
)

print("\n阈值表列名：")
print(threshold_table.columns.tolist())
print("\n阈值表前5行：")
print(threshold_table.head().to_string(index=False))

官方相关矩阵大小： (231, 231)
行列名称顺序一致： True

前10个官方名称：
['Acetaldehyde', 'Carbon disulfide', 'Diethyl sulfide', 'Dimethyl sulfide', 'Ethyl 2-methylbutyrate', 'Ethyl acetate', 'Ethyl decanoate', 'Ethyl hexanoate', 'Ethyl 3-methylbutyrate', 'Ethyl octanoate']

阈值表列名：
['field_name_python', 'field_name_R', 'chemical_name', 'chemical_name_short', 'calibrated', 'origin_paper', 'unit', 'aging', 'off_flavors', 'threshold_mean', 'threshold_median']

阈值表前5行：
   field_name_python          field_name_R    chemical_name chemical_name_short calibrated               origin_paper unit    aging off_flavors  threshold_mean  threshold_median
     2,3-hexanedione      X2.3.hexanedione  2,3-hexanedione     2,3-hexanedione        NaN Internal standard solution  NaN      NaN         NaN             NaN               NaN
    acetic acid.g.L.      acetic_acid.g.L.      Acetic acid         Acetic acid        yes                 Wild flora  g/L      NaN         NaN         0.14025             0.145
        acetophenone  

In [16]:
dictionary = threshold_table.copy()

# 只去除名称首尾空格，不改写化学名称
for col in ["field_name_R", "chemical_name", "chemical_name_short"]:
    dictionary[col] = dictionary[col].astype("string").str.strip()

matched_dict = dictionary.loc[
    dictionary["field_name_R"].isin(feature_cols)
].copy()

missing_fields = [
    name for name in feature_cols
    if name not in set(matched_dict["field_name_R"].dropna())
]

duplicate_fields = matched_dict.loc[
    matched_dict["field_name_R"].duplicated(keep=False),
    ["field_name_R", "chemical_name"]
]

print("匹配到的不同输入字段数：",
      matched_dict["field_name_R"].nunique())
print("\n未匹配的输入字段：")
print(missing_fields)

print("\n重复字段记录：")
print(duplicate_fields.to_string(index=False))

# 查看哪个名称列与官方矩阵标签更一致
official_names = set(official_corr.columns)

for name_col in ["chemical_name", "chemical_name_short"]:
    matched = matched_dict.loc[
        matched_dict[name_col].isin(official_names),
        "field_name_R"
    ].nunique()

    print(f"\n{name_col}能精确对应官方名称的字段数：{matched}")

匹配到的不同输入字段数： 231

未匹配的输入字段：
[]

重复字段记录：
Empty DataFrame
Columns: [field_name_R, chemical_name]
Index: []

chemical_name能精确对应官方名称的字段数：212

chemical_name_short能精确对应官方名称的字段数：209


In [17]:
# 按1.csv字段顺序整理字典
feature_dictionary = matched_dict.set_index(
    "field_name_R"
).loc[feature_cols]

# 字典里不能直接对应官方矩阵的记录
unmatched = feature_dictionary.loc[
    ~feature_dictionary["chemical_name"].isin(official_corr.columns),
    ["chemical_name", "chemical_name_short", "unit"]
]

# 官方矩阵中，尚未被chemical_name匹配的名称
unmatched_official = [
    name for name in official_corr.columns
    if name not in set(feature_dictionary["chemical_name"].dropna())
]

print("字典中未直接匹配的记录：")
print(unmatched.to_string())

print("\n官方矩阵中未匹配的名称：")
for name in unmatched_official:
    print(repr(name))

print("\n字典中重复的chemical_name：")
print(
    feature_dictionary.loc[
        feature_dictionary["chemical_name"].duplicated(keep=False),
        ["chemical_name"]
    ].to_string()
)

字典中未直接匹配的记录：
                                                          chemical_name                chemical_name_short  unit
field_name_R                                                                                                    
alpha.phellandrene                                      Î±-phellandrene                    Î±-phellandrene  A.U.
X.E..beta.ocimene                                          E-Î²-ocimene                       E-Î²-ocimene  mg/L
trans.dihydro.beta.terpineol                 Trans-dihydro-Î²-terpineol         Trans-dihydro-Î²-terpineol   NaN
cis.beta.terpineol                                     Cis-Î²-terpineol                   Cis-Î²-terpineol   NaN
cis.dihydro.beta.terpineol                     Cis-dihydro-Î²-terpineol           Cis-dihydro-Î²-terpineol   NaN
beta.cyclohomocitral                                 Î²-cyclohomocitral                 Î’-cyclohomocitral   NaN
trans.dihydro.alpha.terpinyl.acetate  Trans-dihydro-Î±-terpinyl acetate  Trans-dihy

In [18]:
# 保留原名称，另建修正列
feature_dictionary["official_name"] = (
    feature_dictionary["chemical_name"]
    .str.replace("Î±", "α", regex=False)
    .str.replace("Î²", "β", regex=False)
)

official_order = feature_dictionary["official_name"].tolist()

# 检查是否真正一一对应、完整覆盖
assert feature_dictionary["official_name"].notna().all()
assert feature_dictionary["official_name"].is_unique
assert set(official_order) == set(official_corr.columns)

# 按1.csv字段顺序排列官方矩阵，再换回代码字段名
official_aligned = official_corr.loc[
    official_order, official_order
].copy()

official_aligned.index = feature_cols
official_aligned.columns = feature_cols

# 重算全部231项，便于与官方矩阵完整比较
recomputed = chem[feature_cols].corr(
    method="spearman",
    min_periods=200
)

# 每对只比较一次，排除对角线
mask = np.triu(
    np.ones(recomputed.shape, dtype=bool),
    k=1
)
official_values = official_aligned.to_numpy(dtype=float)[mask]
recomputed_values = recomputed.to_numpy(dtype=float)[mask]

valid = np.isfinite(official_values) & np.isfinite(recomputed_values)
difference = np.abs(
    official_values[valid] - recomputed_values[valid]
)

print("成功映射字段数：", len(official_order))
print("实际比较的指标对：", valid.sum())
print("未能比较的指标对：", (~valid).sum())
print(f"平均绝对差：{difference.mean():.8f}")
print(f"最大绝对差：{difference.max():.8f}")
print("差异超过0.001的对数：", (difference > 0.001).sum())

feature_dictionary.reset_index().to_csv(
    output_dir / "chemical_name_mapping.csv",
    index=False, encoding="utf-8-sig"
)

成功映射字段数： 231
实际比较的指标对： 26565
未能比较的指标对： 0
平均绝对差：0.00000000
最大绝对差：0.00000000
差异超过0.001的对数： 0


In [19]:
# 从官方矩阵中取出此前218个候选
official_sub = official_aligned.loc[compound_cols, compound_cols]

mask = np.triu(
    np.ones(official_sub.shape, dtype=bool),
    k=1
)

official_pairs = (
    official_sub.where(mask)
    .stack()
    .dropna()
    .rename("Spearman相关系数")
    .reset_index()
)
official_pairs.columns = ["指标1", "指标2", "Spearman相关系数"]

official_pairs["相关强度"] = (
    official_pairs["Spearman相关系数"].abs()
)

official_edges = official_pairs.loc[
    official_pairs["相关强度"] >= 0.6
].copy()

# 将每条边表示为无方向的节点组合，比较两份边集合
def edge_set(table):
    return {
        frozenset((a, b))
        for a, b in zip(table["指标1"], table["指标2"])
    }

same_edges = edge_set(official_edges) == edge_set(network_edges)

print("官方矩阵提取的边数：", len(official_edges))
print("与原网络边集合一致：", same_edges)
assert same_edges

official_edges.to_csv(
    output_dir / "network_edges_from_official.csv",
    index=False,
    encoding="utf-8-sig"
)


官方矩阵提取的边数： 245
与原网络边集合一致： True
